# 03 - M7 (7 clases)

Clasificación multiclase con los 12 modelos repetida `N_RUNS` veces (10 por defecto) con splits distintos (tuning con GridSearchCV en la primera corrida), test de Wilcoxon, análisis SHAP y clasificación con las 10 features más importantes.

In [ ]:
from functools import partial

import pandas as pd

from csp_perovskites import config, io, plotting, preprocessing as pp
from csp_perovskites import datasets as ds, evaluation as ev, models, shap_analysis as sa

## Parámetros

In [ ]:
N_RUNS = config.N_RUNS
CV = config.CV_FOLDS
SCALER = "minmax"          # "minmax" (paper), "standard" o None
MODELS = None              # None = los 12 modelos; o una lista de nombres
SHAP_MAX_SAMPLES = None    # submuestreo del test para acelerar SHAP
DROP_POLYMORPHS_FROM_TEST = False  # True = evaluar solo en compuestos con una sola etiqueta

split_fn = partial(ds.make_m7_split, drop_polymorphs_from_test=DROP_POLYMORPHS_FROM_TEST)
df = pp.clean_master(io.load_master())
df_decor, _ = pp.decorrelate(df)

## Clasificación con todas las features

In [ ]:
full = ev.run_repeated(df_decor, split_fn, models.get_classifiers(MODELS),
                       n_runs=N_RUNS, cv=CV, scaler=SCALER)
summary = ev.summarize(full.runs)
io.save_table(full.runs, "M7_runs.csv")
io.save_table(summary, "M7_summary.csv", index=True)
io.save_table(pd.DataFrame({m: [p] for m, p in full.best_params.items()}).T.rename(columns={0: "best_params"}),
              "M7_best_params.csv", index=True)
ev.summarize(full.runs, as_text=True)

## Test de Wilcoxon entre los mejores modelos

In [ ]:
top = [m for m in config.TOP_MODELS if m in full.best_params]
ev.wilcoxon_pairwise(full.runs, top, metric="accuracy")

## SHAP (Random Forest, Extra Trees, LightGBM)

In [ ]:
shap_results = sa.explain_models(split_fn(df_decor, seed=config.RANDOM_STATE), max_samples=SHAP_MAX_SAMPLES)
for model_name, result in shap_results.items():
    fig = sa.plot_shap_summary(result, top_k=10, title=model_name)
    io.save_fig(fig, f"M7_{model_name.replace(' ', '')}_shap_top10.png", directory=config.SHAP_DIR)

combined = sa.combine_importances(shap_results, normalize=True)
io.save_csv(combined, config.SHAP_DIR / "Combined_SHAP_importance_M7.csv")
fig = plotting.top_features_bar(combined, k=10)
io.save_fig(fig, "top10_features_M7.png")

shap_top10 = sa.top_k_features(combined, k=10)
print("Top-10 SHAP:  ", shap_top10)
print("Top-10 paper: ", config.TOP10_M7)

## Clasificación con features reducidas (M7*)

Por defecto se usa la lista del paper (`config.TOP10_M7`); cambia a `shap_top10` para usar la calculada arriba.

In [ ]:
REDUCED_FEATURES = config.TOP10_M7

reduced = ev.run_repeated(df_decor, partial(split_fn, features=REDUCED_FEATURES),
                          models.get_classifiers(config.TOP_MODELS), n_runs=N_RUNS, cv=CV, scaler=SCALER)
io.save_table(reduced.runs, "M7_reduced_runs.csv")

cols = ["accuracy_mean", "accuracy_std", "fit_time_mean", "predict_time_mean"]
comparison = pd.concat({"M7": summary.loc[config.TOP_MODELS, cols],
                        "M7*": ev.summarize(reduced.runs).loc[config.TOP_MODELS, cols]}, axis=1)
io.save_table(comparison, "M7_vs_reduced.csv", index=True)
comparison

## Comparación M4 vs M7 (requiere haber corrido `02_M4.ipynb`)

In [ ]:
m4_path = config.TABLES_DIR / "M4_summary.csv"
if m4_path.exists():
    m4_summary = pd.read_csv(m4_path, index_col=0)
    fig = plotting.metric_bars({"M4": m4_summary, "M7": summary}, metric="accuracy_mean")
    io.save_fig(fig, "classifier_accuracies_M4_M7.png")